# g1_limpo — treino DO ZERO no Colab (`zero03`)

Linhagem NOVA: pesos aleatórios, sem checkpoint de entrada, sem retomada. A sessão roda o
que couber em `HORAS_LIMITE` e para.

**Antes de rodar:** Runtime → Change runtime type → **GPU**. A branch tem de estar no
GitHub. O Drive é só saída; nada a subir antes.

⚠ Este notebook não continua nada. Quem continua uma linhagem é o
`g1_limpo_kaggle.ipynb`, que lê o `model_*.pt` de maior número.

In [ ]:
# ⚠ NADA DE `import torch` aqui: ele registra operadores C++ no import, e se entrar antes
# do pip um reload depois levanta `Only a single TORCH_LIBRARY ... triton`.
import subprocess, sys
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), "sem GPU: Runtime -> Change runtime type -> GPU"

In [ ]:
# ⚠ LISTA DE ARGUMENTOS, nunca string de shell: `numpy<2.5` viraria redirecionamento e o
# pip não rodaria. Só `mjlab` — ele pina a árvore inteira. `torch` NÃO entra: trocá-lo
# perde a GPU sem avisar.
import subprocess, sys
cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "mjlab==1.5.3"]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])
assert r.returncode == 0, "o pip falhou"

In [ ]:
# ⚠ Se o pip trocou o torch por um build sem CUDA, tudo abaixo roda em CPU sem reclamar.
# A checagem vai num subprocesso porque `reload(torch)` não existe.
import subprocess, sys
chk = subprocess.run([sys.executable, "-c",
    "import torch;print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True)
print("subprocesso:", chk.stdout.strip() or chk.stderr[-400:])
assert " True" in chk.stdout, "o pip levou a CUDA embora"

import torch, mjlab, mujoco
print(f"torch {torch.__version__}  {torch.cuda.get_device_name(0)}  |  mujoco {mujoco.__version__}")

In [ ]:
import importlib, os, pathlib, shutil, subprocess, sys
os.environ.setdefault("MUJOCO_GL", "egl")

RUN    = "zero03"              # ⚠ NOME NOVO A CADA MUDANÇA DE CONJUNTO. Ele é a impressão
                               # digital da tabela de recompensa, e mantém duas tabelas
                               # diferentes fora do MESMO gráfico.
BRANCH = "exp/g1-limpo-v2"

# ⚠ CAMINHOS LOCAIS, não no Drive: o `tfevents` é anexado a cada iteração e o FUSE do
# Drive reescreve o arquivo inteiro a cada flush. O Drive recebe CÓPIAS.
BASE     = pathlib.Path("/content")
RAIZ     = BASE / "g1"
LOG_ROOT = BASE / "logs"
raiz_exp = LOG_ROOT / "g1_limpo"

if RAIZ.exists():
    shutil.rmtree(RAIZ)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1",
                "https://github.com/JoaoBornelli/g1_training.git", str(RAIZ)], check=True)
print("clone =", subprocess.run(["git", "-C", str(RAIZ), "log", "--oneline", "-1"],
                                capture_output=True, text=True).stdout.strip())

# ⚠ `invalidate_caches` não é higiene: o Python cacheia um finder POR DIRETÓRIO, e o de um
# diretório que não existia na inserção fica cacheado como VAZIO.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
importlib.invalidate_caches()

from google.colab import drive; drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/g1_limpo"); DRIVE.mkdir(exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

if sorted(raiz_exp.glob(f"*_{RUN}")):
    print(f"\n⚠⚠ JÁ EXISTE run de {RUN!r}: rodar o treino de novo começa OUTRA VEZ do zero,"
          " numa pasta nova, e não continua esta.")
print("log_root =", LOG_ROOT, "| vazio:", not list(raiz_exp.rglob("*")))

In [ ]:
# =====================================================================
#  SAÍDA — a cópia final e a periódica para o Drive.
# =====================================================================
import pathlib, re, shutil, threading, time

ULTIMO_CKPT = None

def _run_nova():
    # a pasta de run mais recente desta `RUN`, e seus `model_*.pt` ordenados por número
    if not raiz_exp.is_dir():
        return None, []
    runs = sorted(p for p in raiz_exp.iterdir() if p.is_dir() and p.name.endswith(RUN))
    if not runs:
        return None, []
    cks = sorted(runs[-1].glob("model_*.pt"),
                 key=lambda p: int(re.search(r"(\d+)", p.name).group(1)))
    return runs[-1], cks

def copia_para_drive():
    # o último checkpoint, o `.pesos.json` e os tfevents vão para `MyDrive/g1_limpo`
    global ULTIMO_CKPT
    nova, cks = _run_nova()
    if not cks:
        print("nada para copiar: o treino não salvou checkpoint")
        return
    ULTIMO_CKPT = cks[-1]
    for p in (ULTIMO_CKPT, raiz_exp / f"{RUN}.pesos.json"):
        if p.exists():
            shutil.copy2(p, DRIVE / p.name)
            print("drive =", DRIVE / p.name)
    for ev in sorted(nova.glob("events.out.tfevents*")):
        shutil.copy2(ev, DRIVE / ev.name)
    print(f"run {nova.name}: {len(cks)} checkpoints, último {ULTIMO_CKPT.name}")

def copia_periodica(intervalo_s=600):
    # ⚠ Contra a QUEDA da sessão: sem isto a perda é tudo desde o começo.
    # ⚠ Só o ÚLTIMO checkpoint, nunca a pasta. E pula um `.pt` escrito há menos de 30 s:
    #   o `torch.save` não é atômico e o arquivo pode estar pela metade.
    def _laco():
        visto = None
        while True:
            time.sleep(intervalo_s)
            try:
                _, cks = _run_nova()
                if not cks:
                    continue
                u = cks[-1]
                chave = (u.name, u.stat().st_mtime)
                if chave == visto or time.time() - chave[1] < 30:
                    continue
                dst = DRIVE / "em_curso" / u.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(u, dst)
                visto = chave
                print(f"[drive] em_curso/{u.name}", flush=True)
            except Exception as e:            # a thread NUNCA derruba o treino
                print(f"[drive] falhou: {e!r}", flush=True)
    threading.Thread(target=_laco, daemon=True, name="copia_drive").start()

In [ ]:
# TENSORBOARD AO VIVO — rode ANTES do treino.
# ⚠ A célula do treino BLOQUEIA o notebook, mas este painel continua se atualizando
#   sozinho enquanto ela roda. Depois dela, só abriria no fim.
# ⚠ O log ainda não existe agora: o painel abre em "No dashboards are active" e se
#   preenche na primeira escrita do runner. Use o botão de recarregar DELE — re-rodar
#   esta célula abre uma segunda instância.
%load_ext tensorboard
%tensorboard --logdir $LOG_ROOT --reload_interval 30

## O treino

Três diferenças em relação ao notebook de continuação: a tabela do `limite_de_junta` é a
**do zero** (a rampa acaba NO batente, e não em 155% do curso), `max_iterations` vem só do
relógio, e a LR fica no default de 1e-3 — baixar para 5e-4 é regra de warm-start, e aqui
não há política velha para desmanchar.

In [ ]:
import dataclasses, inspect, json, sys
sys.path.insert(0, str(RAIZ))
import g1_limpo
from g1_limpo import comando as CMD, knobs as KN, terminacoes as TM_
from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.managers.termination_manager import TerminationTermCfg
from mjlab.managers.scene_entity_config import SceneEntityCfg

NUM_ENVS     = 8192            # ⚠ pelo HOSPEDEIRO, não pela VRAM: medido, a GPU do Colab
                               # roda 8192 e a da Kaggle não, com os mesmos 16 GB.
HORAS_LIMITE = 10.5
SEG_POR_ITER = 9.0             # ⚠ ESTIMATIVA. Leia o `Iteration time` real na 1ª iteração
                               # e corrija: baixo demais e a sessão para cedo; alto demais
                               # e ela não termina.

cfg = dataclasses.replace(TrainConfig.from_task(g1_limpo.TASK_ID), log_root=str(LOG_ROOT))
cfg.env.scene.num_envs   = NUM_ENVS
cfg.agent.run_name       = RUN
cfg.agent.logger         = "tensorboard"
cfg.agent.max_iterations = int(HORAS_LIMITE * 3600 / SEG_POR_ITER)

# ------------------------------------------------- a tabela DO ZERO do limite_de_junta
# ⚠ A rampa acaba NO BATENTE. A tabela larga do knob acaba em 155% do meio-curso: ela
# conserta política viciada e ACEITA o batente como preço. Do zero o robô tem de aprender
# a não chegar nele.
# ⚠⚠ A CINTURA tem tripla própria: o `waist_pitch` tem só 60° de curso e a referência da
# IK o põe em frac 0,900 — com limiar 0,85 esta rampa cobrava A PRÓPRIA REFERÊNCIA, 1,72
# por passo. Com 0,95 ela sai de graça, e o `k` sobe junto senão a mudança AFROUXA o
# freio. Quem cobre o além do teto é a terminação abaixo: as duas são UMA mudança só.
_lj = dataclasses.replace(KN.LimiteDeJunta(),
                          tornozelo=(20.0, 0.15, 0.85), punho=(20.0, 0.15, 0.85),
                          resto=(20.0, 0.15, 0.85),     cintura=(80.0, 0.05, 0.95),
                          hip_yaw=(20.0, 0.15, 0.25))
cfg.env.rewards["limite_de_junta"].params["tabela"] = _lj.por_padrao()

cfg.env.sim.mujoco.cone     = "pyramidal"   # ⚠ "elliptic" já divergiu para NaN duas vezes
cfg.env.sim.mujoco.impratio = 2.0

cfg.env.terminations["batente_da_cintura"] = TerminationTermCfg(
    func=TM_.NoBatente,
    params={"juntas": (".*waist_pitch.*", ".*waist_roll.*", ".*waist_yaw.*"),
            "frac_max": 1.00, "asset_cfg": SceneEntityCfg("robot")})

# ------------------------------------------------- o clone é o que eu penso que é?
# ⚠ Poucos asserts, e cada um pega um modo de falha SILENCIOSO: clone velho que treina
# dez horas com a tabela errada e não avisa.
rw, tm, cu = cfg.env.rewards, cfg.env.terminations, cfg.env.curriculum
_tab = rw["limite_de_junta"].params["tabela"]
assert len(_tab) == 14 and all(len(v) == 3 for v in _tab.values()), \
    f"clone anterior à tripla `(k, teto, limiar)`: {_tab}"
assert len(rw) == 29 and list(rw)[-1] == "renda_congelada", \
    f"{len(rw)} recompensas, última {list(rw)[-1]!r} — a `renda_congelada` lê as outras " \
    "e tem de ser a última"
assert list(cu) == ["command_vel", "forma", "nivel", "elo"], \
    f"ordem do currículo errada ({list(cu)}): `forma` e `nivel` leriam o elo do episódio SEGUINTE"
assert sorted(tm) == ["batente_da_cintura", "caixa_largada", "fell_over", "time_out"], sorted(tm)
assert hasattr(CMD, "FACE_DE_PE"), \
    "clone anterior a 21/09: o BOTAR ainda congela a face na caixa TOMBADA nas mãos, e o " \
    "`alinhado` passa a exigir que ela MANTENHA o tombo"
assert "FACE_CONGELADA" in inspect.getsource(CMD.AlvoCaixaCmd._recalcula_sigmas), \
    "clone anterior a 22/09: o `sigma_ori` do PEGAR ainda vem de um erro que é ZERO por " \
    "construção, e tombar a caixa ao erguê-la sai de graça"
assert {"caixa_na_pega", "caixa_no_botar"} <= set(cfg.env.metrics), \
    "sem as duas sentinelas não dá para ver de qual elo vem o tombo da caixa"
assert cfg.agent.algorithm.learning_rate == 1.0e-3 and cfg.agent.seed == 42
assert not str(LOG_ROOT).startswith(str(RAIZ)), "log dentro do clone: o re-clone o apaga"

print(f"envs {NUM_ENVS} | lote {NUM_ENVS * cfg.agent.num_steps_per_env} | "
      f"{cfg.agent.max_iterations} iterações | LR {cfg.agent.algorithm.learning_rate}")
print("[CONFERE] ok. começando do zero.\n")

raiz_exp.mkdir(parents=True, exist_ok=True)
(raiz_exp / f"{RUN}.pesos.json").write_text(json.dumps(
    {k: float(v.weight) for k, v in rw.items()}, indent=1, sort_keys=True))

copia_periodica()
try:
    launch_training(g1_limpo.TASK_ID, cfg)
finally:
    copia_para_drive()

## O que olhar no log

| iteração | canal | o que responde |
|---|---|---|
| ~400 | `fell_over` e `postura_ereta` | ele fica de pé? |
| ~1000 | `Metrics/twist/razao_marcha` | passou de 0,50? é a marcha |
| ~2500 | `Curriculum/forma/sorteio` | desceu de 0,95? a manipulação entrou |
| sempre | `Episode_Metrics/caixa_na_pega` | graus de tombo ao SEGURAR. Tem de cair; o fecho exige 25° |
| sempre | `Episode_Metrics/caixa_no_botar` | graus de tombo ao POUSAR. Mesma régua |
| sempre | `Curriculum/forma/s_B` e `s_C` | as taxas de fecho do PEGAR e do BOTAR |
| sempre | `Episode_Metrics/velocidade_de_junta` | o freio de segurança; era 2,0 antes do peso −15 |
| a 1ª | `Iteration time` | corrija o `SEG_POR_ITER` com o valor real |

**Continuar depois:** o `finally` deixou o `model_*.pt` e o `.pesos.json` em
`MyDrive/g1_limpo/`. Se a sessão caiu, o último está em `MyDrive/g1_limpo/em_curso/`.
Abra o `g1_limpo_kaggle.ipynb` e troque o `RUN` de lá para `zero03`.